<a href="https://colab.research.google.com/github/amaldevk/AI-powered-paraphrasing-tool/blob/main/AI_Powered_Paraphrasing_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch language-tool-python sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 3.1 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import language_tool_python
from sentence_transformers import SentenceTransformer, util

In [3]:
model_name = "humarin/chatgpt_paraphraser_on_T5_base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:

grammar_tool = language_tool_python.LanguageTool('en-US')


similarity_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def paraphrase_text(original_text: str) -> str:

    input_prompt = "paraphrase: " + original_text
    inputs = tokenizer(input_prompt, padding="longest", max_length=256, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_length=256,
        num_beams=5,
        temperature=1.2,
        do_sample=True,
        repetition_penalty=2.5,
        no_repeat_ngram_size=2,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [6]:
def check_grammar_and_fluency(text: str):

    matches = grammar_tool.check(text)
    corrected_text = grammar_tool.correct(text)

    categories = {"Grammar": 0, "Spelling": 0, "Fluency/Style": 0}
    for match in matches:
        rule_identifier = getattr(match, 'rule_id', getattr(match, 'ruleId', '')).lower()
        category_name = getattr(match, 'category', '').lower()

        if "spell" in rule_identifier or "typo" in rule_identifier or "spell" in category_name:
            categories["Spelling"] += 1
        elif "grammar" in rule_identifier or "syntax" in rule_identifier or "grammar" in category_name:
            categories["Grammar"] += 1
        else:
            categories["Fluency/Style"] += 1

    return corrected_text, len(matches), categories

In [7]:
def compute_semantic_similarity(original: str, paraphrased: str) -> float:
    emb1 = similarity_model.encode(original, convert_to_tensor=True)
    emb2 = similarity_model.encode(paraphrased, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb1, emb2).item()
    return round(semantic_sim, 4)

In [8]:
def print_evaluation_report(original: str, final_text: str, error_count: int, categories: dict, similarity_score: float):

    print("\n")

    print(f"ORIGINAL TEXT:\n  \"{original}\"\n")
    print(f"PARAPHRASED TEXT:\n  \"{final_text}\"")
    print("\n")
    print("1. QUALITY, GRAMMAR & FLUENCY AUDIT:")
    print(f"   • Total Issues Found & Fixed : {error_count}")
    print(f"   • Grammar Errors             : {categories['Grammar']}")
    print(f"   • Spelling Mistakes          : {categories['Spelling']}")
    print(f"   • Fluency & Style Fixes      : {categories['Fluency/Style']}")
    print("\n")
    print("2. SEMANTIC SIMILARITY METRIC: \n")
    print(f"   • Semantic Similarity Score  : {similarity_score * 100:.1f}%")

In [9]:
while True:
    user_text = input("\nEnter text to paraphrase (or type 'exit' to quit): ").strip()

    if user_text.lower() == 'exit':
        print("\nExit...")
        break

    if not user_text:
        print("Please enter valid text.")
        continue

    raw_paraphrase = paraphrase_text(user_text)

    final_paraphrase, error_count, categories = check_grammar_and_fluency(raw_paraphrase)

    similarity_score = compute_semantic_similarity(user_text, final_paraphrase)

    print_evaluation_report(user_text, final_paraphrase, error_count, categories, similarity_score)


Enter text to paraphrase (or type 'exit' to quit): Machine learning model need huge amount of data to achieve high accuracy. If the training dataset contains many error, the output prediction will be incorrect and unreliable for user.


ORIGINAL TEXT:
  "Machine learning model need huge amount of data to achieve high accuracy. If the training dataset contains many error, the output prediction will be incorrect and unreliable for user."

PARAPHRASED TEXT:
  "The high accuracy of machine learning models is dependent on a significant amount of data. If there are many errors in the training dataset, the output prediction will be inaccurate and unreliable to the user."


1. QUALITY, GRAMMAR & FLUENCY AUDIT:
   • Total Issues Found & Fixed : 0
   • Grammar Errors             : 0
   • Spelling Mistakes          : 0
   • Fluency & Style Fixes      : 0


2. SEMANTIC SIMILARITY METRIC: 

   • Semantic Similarity Score  : 93.3%

Enter text to paraphrase (or type 'exit' to quit): exit

Exit...
